In [4]:
import pandas as pd
import numpy as np
import os
from scipy.stats import linregress
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

CROP_MAP = {
    'PD': 'Paddy (PD)', 
    'MZ': 'Corn (MZ)', 
    'CH': 'Chili (CH)'
}

def build_ml_risk_model():
    print("\n" + "="*65)
    print("--- ML BASED RISK MODEL ---")

    # Inputs
    season_code = input("Enter Season (M/Y): ").upper().strip()
    crop_code = input("Enter Crop Code (PD/BO/CH/SY/MZ): ").upper().strip()

    season_full = "Maha" if season_code == "M" else "Yala"
    excel_col = CROP_MAP.get(crop_code)

    # Load registered data
    reg_file = f"E:\\FYP\\Datasets\\Registered\\{plan} Plan {season_full}.xlsx"
    
    try:
        reg_df = pd.read_excel(reg_file)
        reg_df.set_index('Divisional Secretariat (DS)', inplace=True)
    except FileNotFoundError:
        print(f"Error: Registered data file not found at {reg_file}")
        return

    # ==========================================================
    # MODIFIED SECTION: Load ONLY the two specific Excel files
    # ==========================================================
    
    # Change this path to the exact folder where your two Excel files are located
    folder_path = f"E:\\FYP\\Datasets\\{plan}\\" 
    
    file_2024 = os.path.join(folder_path, f"2024 {season_full} {crop_code}.xlsx")
    file_2025 = os.path.join(folder_path, f"2025 {season_full} {crop_code}.xlsx")

    all_claims = []

    # Read the 2024 Excel file
    if os.path.exists(file_2024):
        print(f"Loading {file_2024}...")
        df_2024 = pd.read_excel(file_2024)
        df_2024['year'] = 2024
        all_claims.append(df_2024)
    else:
        print(f"Warning: File not found -> {file_2024}")

    # Read the 2025 Excel file
    if os.path.exists(file_2025):
        print(f"Loading {file_2025}...")
        df_2025 = pd.read_excel(file_2025)
        df_2025['year'] = 2025
        all_claims.append(df_2025)
    else:
        print(f"Warning: File not found -> {file_2025}")

    if not all_claims:
        print("Error: Neither 2024 nor 2025 data files were loaded. Exiting.")
        return

    master_df = pd.concat(all_claims, ignore_index=True)
    
    # ==========================================================

    years_list = sorted(master_df['year'].unique())

    dataset = []

    # =========================
    # FEATURE ENGINEERING
    # =========================
    for div in reg_df.index:
        exposure = reg_df.loc[div, excel_col]
        if pd.isna(exposure):
            continue
            
        if isinstance(exposure, str):
            exposure = float(exposure.replace(',', ''))

        if exposure <= 0:
            continue

        div_data = master_df[master_df['division'] == div]

        for year in years_list:
            year_data = div_data[div_data['year'] == year]

            damaged = year_data['acres'].sum()

            # TARGET VARIABLE
            damage_ratio = damaged / exposure

            # FEATURES
            f_val = damaged / exposure
            dr_val = year_data[year_data['cause']=='Drought']['acres'].sum() / exposure
            el_val = year_data[year_data['cause']=='Elephants']['acres'].sum() / exposure
            in_val = year_data[year_data['cause']=='Insects']['acres'].sum() / exposure
            fl_val = year_data[year_data['cause']=='Flood']['acres'].sum() / exposure
            fr_val = year_data[year_data['cause']=='Fire']['acres'].sum() / exposure

            # Volatility (use past years only)
            past_data = div_data[div_data['year'] <= year]
            annual_harvests = [
                exposure - past_data[past_data['year']==y]['acres'].sum()
                for y in past_data['year'].unique()
            ]

            if len(annual_harvests) > 1 and np.mean(annual_harvests) > 0:
                vol_val = np.std(annual_harvests) / np.mean(annual_harvests)
            else:
                vol_val = 0

            # Trend
            past_years = sorted(past_data['year'].unique())
            claims_series = [
                past_data[past_data['year']==y]['acres'].sum()
                for y in past_years
            ]

            if len(past_years) > 1:
                slope, _, _, _, _ = linregress(range(len(past_years)), claims_series)
                trend_val = slope / exposure
            else:
                trend_val = 0

            dataset.append({
                'division': div,
                'year': year,
                'freq': f_val,
                'drought': dr_val,
                'elephants': el_val,
                'insects': in_val,
                'flood': fl_val,
                'fire': fr_val,
                'vol': vol_val,
                'trend': trend_val,
                'target': damage_ratio
            })

    df = pd.DataFrame(dataset)
    
    if df.empty:
        print("Error: No data available for feature engineering. Make sure division names match between files.")
        return

    # =========================
    # TRAIN ML MODEL
    # =========================

    features = ['freq', 'drought', 'elephants', 'insects', 'flood', 'fire', 'vol', 'trend']
    X = df[features]
    y = df['target']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    # =========================
    # EVALUATION
    # =========================
    y_pred = model.predict(X_test)

    print("\nModel Performance:")
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print("R2 Score:", r2_score(y_test, y_pred))

    # =========================
    # FINAL RISK SCORE
    # =========================
    df['predicted_risk'] = model.predict(X)

    # Scale to 0.1 – 1.0
    min_val = df['predicted_risk'].min()
    max_val = df['predicted_risk'].max()
    
    if max_val == min_val:
        df['Risk_Score'] = 0.1
    else:
        df['Risk_Score'] = 0.1 + ((df['predicted_risk'] - min_val) / (max_val - min_val)) * 0.9

    # Aggregate per division
    final = df.groupby('division')['Risk_Score'].mean().reset_index()

    # Save
    output_path = f"E:\\FYP\\Reports\\ML_Risk_{plan}_{season_full}_{crop_code}.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final.to_csv(output_path, index=False)

    print("\nTop Risk Divisions:")
    print(final.sort_values(by='Risk_Score', ascending=False).to_string(index=False))

    print("\nSaved to:", output_path)

if __name__ == "__main__":
    build_ml_risk_model()


--- ML BASED RISK MODEL ---


Enter Plan (40000 or 100000):  40000
Enter Season (M/Y):  M
Enter Crop Code (PD/BO/CH/SY/MZ):  PD


Loading E:\FYP\Datasets\40000\2024 Maha PD.xlsx...
Loading E:\FYP\Datasets\40000\2025 Maha PD.xlsx...

Model Performance:
RMSE: 0.019240880722960044
R2 Score: 0.9560245961549187

Top Risk Divisions:
         division  Risk_Score
  Mahawilachchiya    0.955923
      Palugaswewa    0.859211
     Nachchadoowa    0.812126
     Horowpothana    0.668571
        Tirappane    0.644756
   Kebithigollewa    0.606121
        Mihintale    0.595462
    Medawachchiya    0.508005
        Ipalogama    0.473272
         Kekirawa    0.462445
          Galnewa    0.438722
      Rajanganaya    0.421776
         Padaviya    0.411166
         Palagala    0.406347
Kahatagasdigiliya    0.393424
    Nochchiyagama    0.368312
          Rambewa    0.362529
 Galenbindunuwewa    0.327079
 NuwaragamCentral    0.297089
          Thalawa    0.284921
   Thambutthegama    0.259147
    NuwaragamEast    0.131772

Saved to: E:\FYP\Reports\ML_Risk_40000_Maha_PD.csv


In [8]:
import pandas as pd
import numpy as np
import os
import glob
from scipy.stats import linregress
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

CROP_MAP = {
    'PD': 'PD', 
    'CH': 'CH', 
    'MZ': 'MZ'
}

def build_ml_risk_model():
    print("\n" + "="*65)
    print("--- ML BASED RISK MODEL ---")

    # Inputs
    plan = input("Enter Plan (40000 or 100000): ").strip()
    season_code = input("Enter Season (M/Y): ").upper().strip()
    crop_code = input("Enter Crop Code (PD/BO/CH/SY/MZ): ").upper().strip()

    season_full = "Maha" if season_code == "M" else "Yala"
    excel_col = CROP_MAP.get(crop_code)

    # ==========================================================
    # 1. LOAD MULTIPLE YEARS OF REGISTERED DATA DYNAMICALLY
    # ==========================================================
    reg_search_pattern = f"E:\\FYP\\Datasets\\Registered\\* {season_full}.xlsx"
    reg_files = glob.glob(reg_search_pattern)
    
    if not reg_files:
        print(f"\nError: No Registered datasets found matching pattern -> {reg_search_pattern}")
        return

    reg_all_years = []
    print("\nLoading Registered Data files:")
    for file_path in reg_files:
        print(f" -> {os.path.basename(file_path)}")
        temp_reg = pd.read_excel(file_path)
        
        # Standardize division column name
        if 'Divisional Secretariat (DS)' in temp_reg.columns:
            temp_reg.rename(columns={'Divisional Secretariat (DS)': 'division'}, inplace=True)
            
        # Extract year from filename (e.g., "2024 Maha.xlsx" -> 2024)
        year = int(os.path.basename(file_path).split(' ')[0])
        temp_reg['year'] = year
        
        # Clean division names
        temp_reg['division'] = temp_reg['division'].astype(str).str.strip()
        
        # Extract only the needed crop exposure column
        if excel_col in temp_reg.columns:
            # Clean commas and convert to float
            temp_reg[excel_col] = temp_reg[excel_col].astype(str).str.replace(',', '')
            temp_reg[excel_col] = pd.to_numeric(temp_reg[excel_col], errors='coerce').fillna(0)
            
            temp_reg.rename(columns={excel_col: 'exposure'}, inplace=True)
            reg_all_years.append(temp_reg[['year', 'division', 'exposure']])

    if not reg_all_years:
        print(f"\nError: Crop column '{excel_col}' not found in registered data.")
        return
        
    reg_df = pd.concat(reg_all_years, ignore_index=True)

    # ==========================================================
    # 2. LOAD ALL RELATED SEASON CLAIMS DATASETS DYNAMICALLY
    # ==========================================================
    claims_search_pattern = f"E:\\FYP\\Datasets\\{plan}\\* {season_full} {crop_code}.xlsx"
    claims_files = glob.glob(claims_search_pattern)
    
    if not claims_files:
        print(f"\nError: No Claims datasets found matching pattern -> {claims_search_pattern}")
        return
        
    all_claims = []
    print("\nLoading Claims Dataset files:")
    for file_path in claims_files:
        print(f" -> {os.path.basename(file_path)}")
        temp_df = pd.read_excel(file_path)
        
        year = int(os.path.basename(file_path).split(' ')[0]) 
        temp_df['year'] = year
        temp_df['division'] = temp_df['division'].astype(str).str.strip()
        
        all_claims.append(temp_df)

    master_df = pd.concat(all_claims, ignore_index=True)

    # ==========================================================
    # 3. FEATURE ENGINEERING (Year-Specific Mapping)
    # ==========================================================
    print("\nProcessing Features...")
    dataset = []
    
    unique_divisions = reg_df['division'].unique()
    years_list = sorted(master_df['year'].unique())

    for div in unique_divisions:
        div_claims = master_df[master_df['division'] == div]
        div_regs = reg_df[reg_df['division'] == div]

        for year in years_list:
            # Find exposure for THIS specific year
            curr_reg = div_regs[div_regs['year'] == year]
            if curr_reg.empty:
                continue # Skip if no register data for this year
                
            exposure = curr_reg['exposure'].values[0]
            if exposure <= 0:
                continue # Skip divisions with 0 planting

            year_data = div_claims[div_claims['year'] == year]
            damaged = year_data['acres'].sum()

            # TARGET VARIABLE
            damage_ratio = damaged / exposure

            # FEATURES
            f_val = damaged / exposure
            dr_val = year_data[year_data['cause']=='Drought']['acres'].sum() / exposure
            el_val = year_data[year_data['cause']=='Elephants']['acres'].sum() / exposure
            in_val = year_data[year_data['cause']=='Insects']['acres'].sum() / exposure
            fl_val = year_data[year_data['cause']=='Flood']['acres'].sum() / exposure
            fr_val = year_data[year_data['cause']=='Fire']['acres'].sum() / exposure

            # Historical Data for Volatility & Trend (up to current year)
            past_years = sorted(div_claims[div_claims['year'] <= year]['year'].unique())
            annual_harvests = []
            claims_series = []

            for py in past_years:
                py_reg = div_regs[div_regs['year'] == py]
                if not py_reg.empty:
                    py_exp = py_reg['exposure'].values[0]
                    py_dam = div_claims[div_claims['year'] == py]['acres'].sum()
                    if py_exp > 0:
                        annual_harvests.append(py_exp - py_dam)
                        claims_series.append(py_dam)

            # Volatility
            if len(annual_harvests) > 1 and np.mean(annual_harvests) > 0:
                vol_val = np.std(annual_harvests) / np.mean(annual_harvests)
            else:
                vol_val = 0

            # Trend
            if len(claims_series) > 1:
                slope, _, _, _, _ = linregress(range(len(claims_series)), claims_series)
                trend_val = slope / exposure
            else:
                trend_val = 0

            dataset.append({
                'division': div,
                'year': year,
                'freq': f_val,
                'drought': dr_val,
                'elephants': el_val,
                'insects': in_val,
                'flood': fl_val,
                'fire': fr_val,
                'vol': vol_val,
                'trend': trend_val,
                'target': damage_ratio
            })

    df = pd.DataFrame(dataset)
    
    if df.empty:
        print("Error: No data available for feature engineering. Check if division names match.")
        return

    # =========================
    # TRAIN ML MODEL
    # =========================
    print("Training Random Forest Model...")
    features = ['freq', 'drought', 'elephants', 'insects', 'flood', 'fire', 'vol', 'trend']
    X = df[features]
    y = df['target']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    # =========================
    # EVALUATION
    # =========================
    y_pred = model.predict(X_test)

    print("\nModel Performance:")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
    print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")

    # =========================
    # FINAL RISK SCORE
    # =========================
    df['predicted_risk'] = model.predict(X)

    # Scale to 0.1 – 1.0
    min_val = df['predicted_risk'].min()
    max_val = df['predicted_risk'].max()
    
    if max_val == min_val:
        df['Risk_Score'] = 0.1
    else:
        df['Risk_Score'] = 0.1 + ((df['predicted_risk'] - min_val) / (max_val - min_val)) * 0.9

    # Aggregate per division
    final = df.groupby('division')['Risk_Score'].mean().reset_index()

    # Save
    output_path = f"E:\\FYP\\Reports\\ML_Risk_{plan}_{season_full}_{crop_code}.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final.to_csv(output_path, index=False)

    # ==========================================================
    # ANURADHAPURA SPECIFIC RANKING
    # ==========================================================
    anuradhapura_ds = [
        'galenbindunuwewa', 'galnewa', 'horowpothana', 'ipalogama', 
        'kahatagasdigiliya', 'kebithigollewa', 'kekirawa', 'mahawilachchiya', 
        'medawachchiya', 'mihinthale', 'mihintale', 'nachchadoowa', 'nochchiyagama', 
        'nuwaragam palatha central', 'nuwaragam palatha east', 'padaviya', 
        'palagala', 'palugaswewa', 'rajanganaya', 'rambewa', 'thalawa', 
        'thambuththegama', 'thirappane'
    ]

    final['temp_lower'] = final['division'].str.lower().str.strip()
    anu_final = final[final['temp_lower'].isin(anuradhapura_ds)].copy()
    anu_final.drop(columns=['temp_lower'], inplace=True)

    print("\n" + "="*65)
    print("--- ANURADHAPURA DISTRICT HIGH RISK RANKING ---")
    
    if not anu_final.empty:
        anu_final = anu_final.sort_values(by='Risk_Score', ascending=False).reset_index(drop=True)
        anu_final['Risk_Score'] = anu_final['Risk_Score'].round(4)
        anu_final.index += 1
        anu_final.index.name = 'Rank'
        print(anu_final.to_string())
    else:
        print("No data found for Anuradhapura divisions in the provided files.")
        
    print("=================================================================")
    print(f"\nFull island-wide report saved to: {output_path}")

if __name__ == "__main__":
    build_ml_risk_model()


--- ML BASED RISK MODEL ---


Enter Plan (40000 or 100000):  40000
Enter Season (M/Y):  M
Enter Crop Code (PD/BO/CH/SY/MZ):  PD



Loading Registered Data files:
 -> 2024 Maha.xlsx
 -> 2025 Maha.xlsx

Loading Claims Dataset files:
 -> 2024 Maha PD.xlsx
 -> 2025 Maha PD.xlsx

Processing Features...
Training Random Forest Model...

Model Performance:
RMSE: 0.0184
R2 Score: 0.9503

--- ANURADHAPURA DISTRICT HIGH RISK RANKING ---
               division  Risk_Score
Rank                               
1       Mahawilachchiya      0.9599
2           Palugaswewa      0.8582
3          Nachchadoowa      0.7525
4          Horowpothana      0.6726
5        Kebithigollewa      0.5833
6             Mihintale      0.5768
7         Medawachchiya      0.4958
8             Ipalogama      0.4591
9              Kekirawa      0.4502
10              Galnewa      0.4197
11          Rajanganaya      0.4156
12             Padaviya      0.4034
13             Palagala      0.3923
14    Kahatagasdigiliya      0.3808
15        Nochchiyagama      0.3594
16              Rambewa      0.3518
17     Galenbindunuwewa      0.3155
18              

In [11]:
import pandas as pd
import numpy as np
import os
import glob
from scipy.stats import linregress
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

CROP_MAP = {
    'PD': 'PD', 
    'CH': 'CH', 
    'MZ': 'MZ'
}

def build_ml_risk_model():
    print("\n" + "="*65)
    print("--- ML BASED RISK MODEL ---")

    # Inputs
    plan = input("Enter Plan (40000 or 100000): ").strip()
    season_code = input("Enter Season (M/Y): ").upper().strip()
    crop_code = input("Enter Crop Code (PD/BO/CH/SY/MZ): ").upper().strip()

    season_full = "Maha" if season_code == "M" else "Yala"
    excel_col = CROP_MAP.get(crop_code)

    # ==========================================================
    # 1. LOAD MULTIPLE YEARS OF REGISTERED DATA DYNAMICALLY
    # ==========================================================
    reg_search_pattern = f"E:\\FYP\\Datasets\\Registered\\* {season_full}.xlsx"
    reg_files = glob.glob(reg_search_pattern)
    
    if not reg_files:
        print(f"\nError: No Registered datasets found matching pattern -> {reg_search_pattern}")
        return

    reg_all_years = []
    print("\nLoading Registered Data files:")
    for file_path in reg_files:
        print(f" -> {os.path.basename(file_path)}")
        temp_reg = pd.read_excel(file_path)
        
        # Standardize division column name
        if 'Divisional Secretariat (DS)' in temp_reg.columns:
            temp_reg.rename(columns={'Divisional Secretariat (DS)': 'division'}, inplace=True)
            
        # Extract year from filename (e.g., "2024 Maha.xlsx" -> 2024)
        year = int(os.path.basename(file_path).split(' ')[0])
        temp_reg['year'] = year
        
        # Clean division names
        temp_reg['division'] = temp_reg['division'].astype(str).str.strip()
        
        # Extract only the needed crop exposure column
        if excel_col in temp_reg.columns:
            # Clean commas and convert to float
            temp_reg[excel_col] = temp_reg[excel_col].astype(str).str.replace(',', '')
            temp_reg[excel_col] = pd.to_numeric(temp_reg[excel_col], errors='coerce').fillna(0)
            
            temp_reg.rename(columns={excel_col: 'exposure'}, inplace=True)
            reg_all_years.append(temp_reg[['year', 'division', 'exposure']])

    if not reg_all_years:
        print(f"\nError: Crop column '{excel_col}' not found in registered data.")
        return
        
    reg_df = pd.concat(reg_all_years, ignore_index=True)

    # ==========================================================
    # 2. LOAD ALL RELATED SEASON CLAIMS DATASETS DYNAMICALLY
    # ==========================================================
    claims_search_pattern = f"E:\\FYP\\Datasets\\{plan}\\* {season_full} {crop_code}.xlsx"
    claims_files = glob.glob(claims_search_pattern)
    
    if not claims_files:
        print(f"\nError: No Claims datasets found matching pattern -> {claims_search_pattern}")
        return
        
    all_claims = []
    print("\nLoading Claims Dataset files:")
    for file_path in claims_files:
        print(f" -> {os.path.basename(file_path)}")
        temp_df = pd.read_excel(file_path)
        
        year = int(os.path.basename(file_path).split(' ')[0]) 
        temp_df['year'] = year
        temp_df['division'] = temp_df['division'].astype(str).str.strip()
        
        all_claims.append(temp_df)

    master_df = pd.concat(all_claims, ignore_index=True)

    # ==========================================================
    # 3. FEATURE ENGINEERING (Year-Specific Mapping)
    # ==========================================================
    print("\nProcessing Features...")
    dataset = []
    
    unique_divisions = reg_df['division'].unique()
    years_list = sorted(master_df['year'].unique())

    for div in unique_divisions:
        div_claims = master_df[master_df['division'] == div]
        div_regs = reg_df[reg_df['division'] == div]

        for year in years_list:
            # Find exposure for THIS specific year
            curr_reg = div_regs[div_regs['year'] == year]
            if curr_reg.empty:
                continue # Skip if no register data for this year
                
            exposure = curr_reg['exposure'].values[0]
            if exposure <= 0:
                continue # Skip divisions with 0 planting

            year_data = div_claims[div_claims['year'] == year]
            damaged = year_data['acres'].sum()

            # TARGET VARIABLE
            damage_ratio = damaged / exposure

            # FEATURES
            f_val = damaged / exposure
            dr_val = year_data[year_data['cause']=='Drought']['acres'].sum() / exposure
            el_val = year_data[year_data['cause']=='Elephants']['acres'].sum() / exposure
            in_val = year_data[year_data['cause']=='Insects']['acres'].sum() / exposure
            fl_val = year_data[year_data['cause']=='Flood']['acres'].sum() / exposure
            fr_val = year_data[year_data['cause']=='Fire']['acres'].sum() / exposure

            # Historical Data for Volatility & Trend (up to current year)
            past_years = sorted(div_claims[div_claims['year'] <= year]['year'].unique())
            annual_harvests = []
            claims_series = []

            for py in past_years:
                py_reg = div_regs[div_regs['year'] == py]
                if not py_reg.empty:
                    py_exp = py_reg['exposure'].values[0]
                    py_dam = div_claims[div_claims['year'] == py]['acres'].sum()
                    if py_exp > 0:
                        annual_harvests.append(py_exp - py_dam)
                        claims_series.append(py_dam)

            # Volatility
            if len(annual_harvests) > 1 and np.mean(annual_harvests) > 0:
                vol_val = np.std(annual_harvests) / np.mean(annual_harvests)
            else:
                vol_val = 0

            # Trend
            if len(claims_series) > 1:
                slope, _, _, _, _ = linregress(range(len(claims_series)), claims_series)
                trend_val = slope / exposure
            else:
                trend_val = 0

            dataset.append({
                'division': div,
                'year': year,
                'freq': f_val,
                'drought': dr_val,
                'elephants': el_val,
                'insects': in_val,
                'flood': fl_val,
                'fire': fr_val,
                'vol': vol_val,
                'trend': trend_val,
                'target': damage_ratio
            })

    df = pd.DataFrame(dataset)
    
    if df.empty:
        print("Error: No data available for feature engineering. Check if division names match.")
        return

    # =========================
    # TRAIN ML MODEL
    # =========================
    print("Training Random Forest Model...")
    features = ['freq', 'drought', 'elephants', 'insects', 'flood', 'fire', 'vol', 'trend']
    X = df[features]
    y = df['target']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    # =========================
    # EVALUATION
    # =========================
    y_pred = model.predict(X_test)

    print("\nModel Performance:")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
    print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")

    # =========================
    # FINAL RISK SCORE
    # =========================
    df['predicted_risk'] = model.predict(X)

    # Normalized Scaling to 0.75 – 1.25 range
    min_val = df['predicted_risk'].min()
    max_val = df['predicted_risk'].max()
    
    if max_val == min_val:
        # Default to 1.0 (baseline) if all divisions have the same risk
        df['Risk_Score'] = 1.0
    else:
        # Scale range is exactly 0.5 (from 0.75 to 1.25)
        df['Risk_Score'] = 0.75 + ((df['predicted_risk'] - min_val) / (max_val - min_val)) * 0.5

    # Aggregate per division
    final = df.groupby('division')['Risk_Score'].mean().reset_index()

    # Save island-wide CSV
    output_path = f"E:\\FYP\\Reports\\ML_Risk_{plan}_{season_full}_{crop_code}.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final.to_csv(output_path, index=False)

    # ==========================================================
    # ANURADHAPURA SPECIFIC RANKING
    # ==========================================================
    anuradhapura_ds = [
        'galenbindunuwewa', 'galnewa', 'horowpothana', 'ipalogama', 
        'kahatagasdigiliya', 'kebithigollewa', 'kekirawa', 'mahawilachchiya', 
        'medawachchiya', 'mihinthale', 'mihintale', 'nachchadoowa', 'nochchiyagama', 
        'nuwaragam palatha central', 'nuwaragam palatha east', 'padaviya', 
        'palagala', 'palugaswewa', 'rajanganaya', 'rambewa', 'thalawa', 
        'thambuththegama', 'thirappane'
    ]

    # Use a temporary lowercase column for matching
    final['temp_lower'] = final['division'].str.lower().str.strip()
    anu_final = final[final['temp_lower'].isin(anuradhapura_ds)].copy()
    anu_final.drop(columns=['temp_lower'], inplace=True)

    print("\n" + "="*65)
    print("--- ANURADHAPURA DISTRICT HIGH RISK RANKING ---")
    
    if not anu_final.empty:
        # Sort Highest Risk to Lowest
        anu_final = anu_final.sort_values(by='Risk_Score', ascending=False).reset_index(drop=True)
        # Format the Risk Score for terminal readability
        anu_final['Risk_Score'] = anu_final['Risk_Score'].round(4)
        # Rank numbers from 1 downwards
        anu_final.index += 1
        anu_final.index.name = 'Rank'
        
        print(anu_final.to_string())
    else:
        print("No data found for Anuradhapura divisions in the provided files.")
        
    print("=================================================================")
    print(f"\nFull island-wide report saved to: {output_path}")

if __name__ == "__main__":
    build_ml_risk_model()


--- ML BASED RISK MODEL ---


Enter Plan (40000 or 100000):  40000
Enter Season (M/Y):  M
Enter Crop Code (PD/BO/CH/SY/MZ):  PD



Loading Registered Data files:
 -> 2024 Maha.xlsx
 -> 2025 Maha.xlsx

Loading Claims Dataset files:
 -> 2024 Maha PD.xlsx
 -> 2025 Maha PD.xlsx

Processing Features...
Training Random Forest Model...

Model Performance:
RMSE: 0.0191
R2 Score: 0.9565

--- ANURADHAPURA DISTRICT HIGH RISK RANKING ---
               division  Risk_Score
Rank                               
1       Mahawilachchiya      1.2320
2           Palugaswewa      1.1713
3          Nachchadoowa      1.1355
4          Horowpothana      1.0657
5        Kebithigollewa      1.0221
6             Mihintale      1.0165
7         Medawachchiya      0.9712
8             Ipalogama      0.9524
9              Kekirawa      0.9475
10              Galnewa      0.9321
11          Rajanganaya      0.9274
12             Padaviya      0.9188
13             Palagala      0.9166
14    Kahatagasdigiliya      0.9102
15        Nochchiyagama      0.8964
16              Rambewa      0.8919
17     Galenbindunuwewa      0.8747
18              

In [13]:
pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 466.4 kB/s eta 0:03:37
   ---------------------------------------- 0.5/101.7 MB 466.4 kB/s eta 0:03:37
   ---------------------------------------- 0.8/101.7 MB 493.7 kB/s eta 0:03:25
   ---------------------------------------- 0.8/101.7 MB 493.7 kB/s eta 0:03:25
   ---------------------------------------- 0.8/101.7 MB 493.7 kB/s eta 0:03:25
   ---------------------------------------- 1.0/101.7 MB 470.4 kB/s eta 0:03:34
   --------------------

In [38]:
import pandas as pd
import numpy as np
import os
import glob
from scipy.stats import linregress
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

def build_ml_risk_model():
    print("\n" + "="*70)
    print("--- ML BASED RISK MODEL (DYNAMIC XGBOOST) ---")
    print("="*70)

    # ==========================================================
    # INPUT
    # ==========================================================
    year_input = input("Enter Year(s) (e.g., 2024,2025 or ALL): ").upper().strip()
    season_input = input("Enter Season(s) (M, Y, ALL): ").upper().strip()
    crop_input = input("Enter Crop(s) (PD, BO, CH, SY, MZ or ALL): ").upper().strip()

    # Parse
    target_years = None if year_input == 'ALL' else [int(y) for y in year_input.split(',')]

    if season_input == 'ALL':
        target_seasons = ['Maha', 'Yala']
    else:
        target_seasons = ['Maha' if s=='M' else 'Yala' for s in season_input.split(',')]

    target_crops = ['PD','BO','CH','SY','MZ'] if crop_input == 'ALL' else crop_input.split(',')

    # ==========================================================
    # LOAD REGISTERED DATA
    # ==========================================================
    reg_all = []
    base_reg_path = "E:\\FYP\\Datasets\\Registered"

    for s in target_seasons:
        season_initial = 'M' if s == 'Maha' else 'Y'

        patterns = [
            os.path.join(base_reg_path, f"* {season_initial}.xlsx"),
            os.path.join(base_reg_path, f"* {s}.xlsx")
        ]

        for pattern in patterns:
            for file in glob.glob(pattern):
                try:
                    year = int(os.path.basename(file).split(' ')[0])
                except:
                    continue

                if target_years and year not in target_years:
                    continue

                df = pd.read_excel(file)

                if 'Divisional Secretariat (DS)' in df.columns:
                    df.rename(columns={'Divisional Secretariat (DS)': 'division'}, inplace=True)

                df['year'] = year
                df['season'] = s
                df['division'] = df['division'].astype(str).str.strip()

                for c in target_crops:
                    df[c] = pd.to_numeric(df.get(c, 0), errors='coerce').fillna(0)

                reg_all.append(df)

    reg_df = pd.concat(reg_all, ignore_index=True)

    # ==========================================================
    # LOAD CLAIMS DATA
    # ==========================================================
    all_claims = []
    base_claims_path = "E:\\FYP\\Datasets"

    if target_years is None:
        years_to_search = [int(y) for y in os.listdir(base_claims_path) if y.isdigit()]
    else:
        years_to_search = target_years

    for year in years_to_search:
        year_folder = os.path.join(base_claims_path, str(year))
        if not os.path.exists(year_folder):
            continue

        for s in target_seasons:
            season_initial = 'M' if s == 'Maha' else 'Y'

            for c in target_crops:
                paths = [
                    os.path.join(year_folder, f"{season_initial} {c}.xlsx"),
                    os.path.join(year_folder, f"{s} {c}.xlsx")
                ]

                for path in paths:
                    if os.path.exists(path):
                        df = pd.read_excel(path)

                        if 'division' not in df.columns:
                            continue

                        df['division'] = df['division'].astype(str).str.strip()
                        df['year'] = year
                        df['season'] = s
                        df['crop'] = c

                        all_claims.append(df)
                        break

    master_df = pd.concat(all_claims, ignore_index=True)

    # ==========================================================
    # FEATURE ENGINEERING
    # ==========================================================
    dataset = []

    for _, reg_row in reg_df.iterrows():
        div = reg_row['division']
        year = reg_row['year']
        season = reg_row['season']

        for crop in target_crops:
            exposure = reg_row[crop]
            if exposure <= 0:
                continue

            claims = master_df[
                (master_df['division'] == div) &
                (master_df['year'] == year) &
                (master_df['season'] == season) &
                (master_df['crop'] == crop)
            ]

            damaged = claims['acres'].sum()
            damaged = min(damaged, exposure)

            damage_ratio = damaged / exposure if exposure > 0 else 0

            # Causes
            def safe_ratio(cause):
                val = claims[claims['cause'] == cause]['acres'].sum()
                return val / exposure if exposure > 0 else 0

            drought = safe_ratio('Drought')
            elephants = safe_ratio('Elephants')
            insects = safe_ratio('Insects')
            flood = safe_ratio('Flood')
            fire = safe_ratio('Fire')

            # Additional features
            claim_count = len(claims)
            severity = damaged

            dataset.append({
                'division': div,
                'year': year,
                'season': season,
                'crop': crop,
                'drought': drought,
                'elephants': elephants,
                'insects': insects,
                'flood': flood,
                'fire': fire,
                'claim_count': claim_count,
                'severity': severity,
                'target': damage_ratio
            })

    df = pd.DataFrame(dataset)

    df = pd.get_dummies(df, columns=['season', 'crop'])

    # ==========================================================
    # MODEL
    # ==========================================================
    features = [c for c in df.columns if c not in ['division','year','target']]
    X = df[features].astype(float)
    y = df['target']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    # ==========================================================
    # EVALUATION
    # ==========================================================
    pred = model.predict(X_test)

    print("\nPerformance:")
    print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
    print("R2:", r2_score(y_test, pred))

    # ==========================================================
    # RISK SCORE
    # ==========================================================
    df['pred'] = model.predict(X)

    min_v = df['pred'].min()
    max_v = df['pred'].max()

    df['Risk_Score'] = 0.75 + ((df['pred'] - min_v) / (max_v - min_v)) * 0.4

    # Weighted aggregation
    df['weight'] = df['severity'] + 1

    final = df.groupby('division').apply(
        lambda x: np.average(x['Risk_Score'], weights=x['weight'])
    ).reset_index(name='Risk_Score')

    # ==========================================================
    # SAVE
    # ==========================================================
    output_path = f"E:\\FYP\\Reports\\ML_Risk_{year_input}_{season_input}_{crop_input}.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final.to_csv(output_path, index=False)

    # ==========================================================
    # PRINT ANURADHAPURA
    # ==========================================================
    anuradhapura_ds = [
        'galenbindunuwewa','galnewa','horowpothana','ipalogama',
        'kahatagasdigiliya','kebithigollewa','kekirawa','mahawilachchiya',
        'medawachchiya','mihintale','nachchadoowa','nochchiyagama',
        'nuwaragam palatha central','nuwaragam palatha east','padaviya',
        'palagala','palugaswewa','rajanganaya','rambewa','thalawa',
        'thambuththegama','thirappane'
    ]

    final['temp'] = final['division'].str.lower().str.strip()
    anu = final[final['temp'].isin(anuradhapura_ds)].drop(columns='temp')

    print("\n--- ANURADHAPURA RANKING ---")
    print(anu.sort_values(by='Risk_Score', ascending=False).reset_index(drop=True))

    print(f"\nSaved to: {output_path}")

if __name__ == "__main__":
    build_ml_risk_model()


--- ML BASED RISK MODEL (DYNAMIC XGBOOST) ---


Enter Year(s) (e.g., 2024,2025 or ALL):  ALL
Enter Season(s) (M, Y, ALL):  M
Enter Crop(s) (PD, BO, CH, SY, MZ or ALL):  PD



Performance:
RMSE: 0.032128054552863376
R2: 0.882710952047313

--- ANURADHAPURA RANKING ---
             division  Risk_Score
0     Mahawilachchiya    1.945521
1         Palugaswewa    1.774582
2        Nachchadoowa    1.614630
3        Horowpothana    1.503930
4      Kebithigollewa    1.437485
5           Mihintale    1.425664
6       Medawachchiya    1.321137
7            Kekirawa    1.297420
8           Ipalogama    1.279121
9         Rajanganaya    1.242210
10            Galnewa    1.224406
11           Padaviya    1.208423
12  Kahatagasdigiliya    1.194398
13           Palagala    1.192063
14      Nochchiyagama    1.146748
15            Rambewa    1.136619
16   Galenbindunuwewa    1.101151
17            Thalawa    1.087946

Saved to: E:\FYP\Reports\ML_Risk_ALL_M_PD.csv


C:\Users\USER\AppData\Local\Temp\ipykernel_3864\1448157638.py:217: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final = df.groupby('division').apply(


In [39]:
import pandas as pd
import numpy as np
import os
import glob
from scipy.stats import linregress
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

def build_ml_risk_model():
    print("\n" + "="*70)
    print("--- ML BASED RISK MODEL (DYNAMIC XGBOOST) ---")
    print("="*70)

    # ==========================================================
    # INPUT
    # ==========================================================
    year_input = input("Enter Year(s) (e.g., 2024,2025 or ALL): ").upper().strip()
    season_input = input("Enter Season(s) (M, Y, ALL): ").upper().strip()
    crop_input = input("Enter Crop(s) (PD, CH or ALL): ").upper().strip()

    # Parse inputs (safely removing accidental spaces before splitting)
    target_years = None if year_input == 'ALL' else [int(y) for y in year_input.replace(' ', '').split(',')]

    if season_input == 'ALL':
        target_seasons = ['Maha', 'Yala']
    else:
        target_seasons = ['Maha' if s=='M' else 'Yala' for s in season_input.replace(' ', '').split(',')]

    target_crops = ['PD','CH'] if crop_input == 'ALL' else crop_input.replace(' ', '').split(',')

    # ==========================================================
    # LOAD REGISTERED DATA
    # ==========================================================
    reg_all = []
    base_reg_path = "E:\\FYP\\Datasets\\Registered"
    print("\nLoading Registered Data...")

    for s in target_seasons:
        season_initial = 'M' if s == 'Maha' else 'Y'

        patterns = [
            os.path.join(base_reg_path, f"* {season_initial}.xlsx"),
            os.path.join(base_reg_path, f"* {s}.xlsx")
        ]

        for pattern in patterns:
            for file in glob.glob(pattern):
                try:
                    year = int(os.path.basename(file).split(' ')[0])
                except:
                    continue

                if target_years and year not in target_years:
                    continue

                df = pd.read_excel(file)

                if 'Divisional Secretariat (DS)' in df.columns:
                    df.rename(columns={'Divisional Secretariat (DS)': 'division'}, inplace=True)

                df['year'] = year
                df['season'] = s
                df['division'] = df['division'].astype(str).str.strip()

                # === THE FIX IS HERE ===
                for c in target_crops:
                    if c in df.columns:
                        # Convert to string to remove commas, then to numeric
                        df[c] = df[c].astype(str).str.replace(',', '')
                        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
                    else:
                        # If crop column doesn't exist, assign 0 directly
                        df[c] = 0

                reg_all.append(df)

    if not reg_all:
        print("Error: No Registered Datasets found matching your criteria.")
        return
        
    reg_df = pd.concat(reg_all, ignore_index=True)

    # ==========================================================
    # LOAD CLAIMS DATA
    # ==========================================================
    all_claims = []
    base_claims_path = "E:\\FYP\\Datasets"
    print("Loading Claims Data...")

    if target_years is None:
        years_to_search = [int(y) for y in os.listdir(base_claims_path) if y.isdigit()]
    else:
        years_to_search = target_years

    for year in years_to_search:
        year_folder = os.path.join(base_claims_path, str(year))
        if not os.path.exists(year_folder):
            continue

        for s in target_seasons:
            season_initial = 'M' if s == 'Maha' else 'Y'

            for c in target_crops:
                paths = [
                    os.path.join(year_folder, f"{season_initial} {c}.xlsx"),
                    os.path.join(year_folder, f"{s} {c}.xlsx")
                ]

                for path in paths:
                    if os.path.exists(path):
                        df = pd.read_excel(path)

                        if 'division' not in df.columns:
                            continue

                        df['division'] = df['division'].astype(str).str.strip()
                        df['year'] = year
                        df['season'] = s
                        df['crop'] = c

                        all_claims.append(df)
                        break

    if not all_claims:
        print("Error: No Claims Datasets found matching your criteria.")
        return
        
    master_df = pd.concat(all_claims, ignore_index=True)

    # ==========================================================
    # FEATURE ENGINEERING
    # ==========================================================
    print("Processing Features...")
    dataset = []

    for _, reg_row in reg_df.iterrows():
        div = reg_row['division']
        year = reg_row['year']
        season = reg_row['season']

        for crop in target_crops:
            exposure = reg_row[crop]
            if exposure <= 0:
                continue

            claims = master_df[
                (master_df['division'] == div) &
                (master_df['year'] == year) &
                (master_df['season'] == season) &
                (master_df['crop'] == crop)
            ]

            damaged = claims['acres'].sum()
            damaged = min(damaged, exposure)

            damage_ratio = damaged / exposure if exposure > 0 else 0

            # Causes
            def safe_ratio(cause):
                val = claims[claims['cause'] == cause]['acres'].sum()
                return val / exposure if exposure > 0 else 0

            drought = safe_ratio('Drought')
            elephants = safe_ratio('Elephants')
            insects = safe_ratio('Insects')
            flood = safe_ratio('Flood')
            fire = safe_ratio('Fire')

            # Additional features
            claim_count = len(claims)
            severity = damaged

            dataset.append({
                'division': div,
                'year': year,
                'season': season,
                'crop': crop,
                'drought': drought,
                'elephants': elephants,
                'insects': insects,
                'flood': flood,
                'fire': fire,
                'claim_count': claim_count,
                'severity': severity,
                'target': damage_ratio
            })

    df = pd.DataFrame(dataset)

    if df.empty:
        print("Error: No usable data could be joined. Check if division names match.")
        return

    df = pd.get_dummies(df, columns=['season', 'crop'])

    # ==========================================================
    # MODEL
    # ==========================================================
    print("Training Model...")
    features = [c for c in df.columns if c not in ['division','year','target']]
    X = df[features].astype(float)
    y = df['target']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    # ==========================================================
    # EVALUATION
    # ==========================================================
    pred = model.predict(X_test)

    print("\nPerformance:")
    print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
    print("R2:", r2_score(y_test, pred))

    # ==========================================================
    # RISK SCORE
    # ==========================================================
    df['pred'] = model.predict(X)

    min_v = df['pred'].min()
    max_v = df['pred'].max()

    if max_v == min_v:
        df['Risk_Score'] = 1.0  
    else:
        df['Risk_Score'] = 0.8 + ((df['pred'] - min_v) / (max_v - min_v)) * 0.4

    # Weighted aggregation
    df['weight'] = df['severity'] + 1

    final = df.groupby('division').apply(
        lambda x: np.average(x['Risk_Score'], weights=x['weight'])
    ).reset_index(name='Risk_Score')

    # ==========================================================
    # SAVE
    # ==========================================================
    y_str = year_input.replace(',','-').replace(' ','')
    s_str = season_input.replace(',','-').replace(' ','')
    c_str = crop_input.replace(',','-').replace(' ','')
    
    output_path = f"E:\\FYP\\Reports\\ML_Risk_{y_str}_{s_str}_{c_str}.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final.to_csv(output_path, index=False)

    # ==========================================================
    # PRINT ANURADHAPURA
    # ==========================================================
    anuradhapura_ds = [
        'galenbindunuwewa','galnewa','horowpothana','ipalogama',
        'kahatagasdigiliya','kebithigollewa','kekirawa','mahawilachchiya',
        'medawachchiya','mihintale','mihinthale','nachchadoowa','nochchiyagama',
        'nuwaragam palatha central','nuwaragam palatha east','padaviya',
        'palagala','palugaswewa','rajanganaya','rambewa','thalawa',
        'thambuththegama','thirappane'
    ]

    final['temp'] = final['division'].str.lower().str.strip()
    anu = final[final['temp'].isin(anuradhapura_ds)].drop(columns='temp')

    print("\n--- ANURADHAPURA RANKING ---")
    
    if not anu.empty:
        anu = anu.sort_values(by='Risk_Score', ascending=False).reset_index(drop=True)
        anu['Risk_Score'] = anu['Risk_Score'].round(4)
        anu.index += 1
        anu.index.name = 'Rank'
        print(anu)
    else:
        print("No matches for Anuradhapura found in the current datasets.")

    print(f"\nSaved to: {output_path}")

if __name__ == "__main__":
    build_ml_risk_model()


--- ML BASED RISK MODEL (DYNAMIC XGBOOST) ---


Enter Year(s) (e.g., 2024,2025 or ALL):  ALL
Enter Season(s) (M, Y, ALL):  M
Enter Crop(s) (PD, CH or ALL):  PD



Loading Registered Data...
Loading Claims Data...
Processing Features...
Training Model...

Performance:
RMSE: 0.020382742363825593
R2: 0.8744762159080974

--- ANURADHAPURA RANKING ---
               division  Risk_Score
Rank                               
1       Mahawilachchiya      1.1818
2           Palugaswewa      1.1275
3          Nachchadoowa      1.1175
4          Horowpothana      1.0604
5        Kebithigollewa      1.0199
6             Mihintale      1.0056
7         Medawachchiya      0.9826
8             Ipalogama      0.9740
9              Kekirawa      0.9617
10          Rajanganaya      0.9560
11             Padaviya      0.9541
12              Galnewa      0.9499
13             Palagala      0.9421
14    Kahatagasdigiliya      0.9378
15        Nochchiyagama      0.9277
16              Rambewa      0.9241
17     Galenbindunuwewa      0.9167
18              Thalawa      0.8976

Saved to: E:\FYP\Reports\ML_Risk_ALL_M_PD.csv


C:\Users\USER\AppData\Local\Temp\ipykernel_3864\2911759537.py:243: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final = df.groupby('division').apply(


In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

def build_ml_risk_model():
    print("\n" + "="*70)
    print("--- ML BASED RISK MODEL (DYNAMIC XGBOOST) ---")
    print("="*70)

    # ==========================================================
    # INPUT
    # ==========================================================
    year_input = input("Enter Year(s) (e.g., 2024,2025 or ALL): ").upper().strip()
    season_input = input("Enter Season(s) (M, Y, ALL): ").upper().strip()
    crop_input = input("Enter Crop(s) (PD, CH or ALL): ").upper().strip()

    target_years = None if year_input == 'ALL' else [int(y) for y in year_input.replace(' ', '').split(',')]

    if season_input == 'ALL':
        target_seasons = ['Maha', 'Yala']
    else:
        target_seasons = ['Maha' if s=='M' else 'Yala' for s in season_input.replace(' ', '').split(',')]

    target_crops = ['PD','CH'] if crop_input == 'ALL' else crop_input.replace(' ', '').split(',')

    # ==========================================================
    # LOAD REGISTERED DATA
    # ==========================================================
    reg_all = []
    base_reg_path = "E:\\FYP\\Datasets\\Registered"
    print("\nLoading Registered Data...")

    for s in target_seasons:
        season_initial = 'M' if s == 'Maha' else 'Y'

        patterns = [
            os.path.join(base_reg_path, f"* {season_initial}.xlsx"),
            os.path.join(base_reg_path, f"* {s}.xlsx")
        ]

        for pattern in patterns:
            for file in glob.glob(pattern):
                try:
                    year = int(os.path.basename(file).split(' ')[0])
                except:
                    continue

                if target_years and year not in target_years:
                    continue

                df = pd.read_excel(file)

                if 'Divisional Secretariat (DS)' in df.columns:
                    df.rename(columns={'Divisional Secretariat (DS)': 'division'}, inplace=True)

                df['year'] = year
                df['season'] = s
                df['division'] = df['division'].astype(str).str.strip()

                for c in target_crops:
                    if c in df.columns:
                        df[c] = df[c].astype(str).str.replace(',', '')
                        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
                    else:
                        df[c] = 0

                reg_all.append(df)

    if not reg_all:
        print("Error: No Registered Data found.")
        return
        
    reg_df = pd.concat(reg_all, ignore_index=True)

    # ==========================================================
    # LOAD CLAIMS DATA
    # ==========================================================
    all_claims = []
    base_claims_path = "E:\\FYP\\Datasets"
    print("Loading Claims Data...")

    years_to_search = target_years if target_years else [int(y) for y in os.listdir(base_claims_path) if y.isdigit()]

    for year in years_to_search:
        year_folder = os.path.join(base_claims_path, str(year))
        if not os.path.exists(year_folder):
            continue

        for s in target_seasons:
            season_initial = 'M' if s == 'Maha' else 'Y'

            for c in target_crops:
                paths = [
                    os.path.join(year_folder, f"{season_initial} {c}.xlsx"),
                    os.path.join(year_folder, f"{s} {c}.xlsx")
                ]

                for path in paths:
                    if os.path.exists(path):
                        df = pd.read_excel(path)

                        if 'division' not in df.columns:
                            continue

                        df['division'] = df['division'].astype(str).str.strip()
                        df['year'] = year
                        df['season'] = s
                        df['crop'] = c

                        all_claims.append(df)
                        break

    if not all_claims:
        print("Error: No Claims Data found.")
        return
        
    master_df = pd.concat(all_claims, ignore_index=True)

    # ==========================================================
    # FEATURE ENGINEERING
    # ==========================================================
    print("Processing Features...")
    dataset = []

    for _, reg_row in reg_df.iterrows():
        div = reg_row['division']
        year = reg_row['year']
        season = reg_row['season']

        for crop in target_crops:
            exposure = reg_row[crop]
            if exposure <= 0:
                continue

            claims = master_df[
                (master_df['division'] == div) &
                (master_df['year'] == year) &
                (master_df['season'] == season) &
                (master_df['crop'] == crop)
            ]

            damaged = min(claims['acres'].sum(), exposure)
            damage_ratio = damaged / exposure if exposure > 0 else 0

            def safe_ratio(cause):
                return claims[claims['cause'] == cause]['acres'].sum() / exposure if exposure > 0 else 0

            dataset.append({
                'division': div,
                'year': year,
                'season': season,
                'crop': crop,
                'drought': safe_ratio('Drought'),
                'elephants': safe_ratio('Elephants'),
                'insects': safe_ratio('Insects'),
                'flood': safe_ratio('Flood'),
                'fire': safe_ratio('Fire'),
                'claim_count': len(claims),
                'severity': damaged,
                'target': damage_ratio
            })

    df = pd.DataFrame(dataset)

    if df.empty:
        print("Error: No usable merged data.")
        return

    df = pd.get_dummies(df, columns=['season', 'crop'])

    # ==========================================================
    # MODEL
    # ==========================================================
    print("Training Model...")

    features = [c for c in df.columns if c not in ['division','year','target']]
    X = df[features].astype(float)
    y = df['target']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = XGBRegressor(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=42
    )

    model.fit(X_train, y_train)

    # ==========================================================
    # EVALUATION
    # ==========================================================
    pred_test = model.predict(X_test)

    print("\nPerformance:")
    print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_test)))
    print("R2:", r2_score(y_test, pred_test))

    # ==========================================================
    # RISK SCORE (FIXED RANGE 0.75 → 1.25)
    # ==========================================================
    df['pred'] = model.predict(X)

    min_v = df['pred'].min()
    max_v = df['pred'].max()

    # Clip outliers (important)
    df['pred'] = np.clip(df['pred'], min_v, max_v)

    if max_v == min_v:
        df['Risk_Score'] = 1.0
    else:
        df['Risk_Score'] = 0.75 + ((df['pred'] - min_v) / (max_v - min_v)) * 0.5

    # ==========================================================
    # AGGREGATION
    # ==========================================================
    df['weight'] = df['severity'] + 1

    final = df.groupby('division').apply(
        lambda x: np.average(x['Risk_Score'], weights=x['weight'])
    ).reset_index(name='Risk_Score')

    # ==========================================================
    # SAVE
    # ==========================================================
    y_str = year_input.replace(',','-').replace(' ','')
    s_str = season_input.replace(',','-').replace(' ','')
    c_str = crop_input.replace(',','-').replace(' ','')

    output_path = f"E:\\FYP\\Reports\\ML_Risk_{y_str}_{s_str}_{c_str}.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    final.to_csv(output_path, index=False)

    # ==========================================================
    # ANURADHAPURA FILTER
    # ==========================================================
    anuradhapura_ds = [
        'galenbindunuwewa','galnewa','horowpothana','ipalogama',
        'kahatagasdigiliya','kebithigollewa','kekirawa','mahawilachchiya',
        'medawachchiya','mihintale','mihinthale','nachchadoowa','nochchiyagama',
        'nuwaragam palatha central','nuwaragam palatha east','padaviya',
        'palagala','palugaswewa','rajanganaya','rambewa','thalawa',
        'thambuththegama','thirappane'
    ]

    final['temp'] = final['division'].str.lower().str.strip()
    anu = final[final['temp'].isin(anuradhapura_ds)].drop(columns='temp')

    print("\n--- ANURADHAPURA RANKING ---")

    if not anu.empty:
        anu = anu.sort_values(by='Risk_Score', ascending=False).reset_index(drop=True)
        anu['Risk_Score'] = anu['Risk_Score'].round(4)
        anu.index += 1
        anu.index.name = 'Rank'
        print(anu)
    else:
        print("No matches found.")

    print(f"\nSaved to: {output_path}")

if __name__ == "__main__":
    build_ml_risk_model()


--- ML BASED RISK MODEL (DYNAMIC XGBOOST) ---
